# DNA-Based Semantic Search: Differentiable Biophysical Representation Learning

**ChemiSearch** — encoding natural language semantics into physical DNA oligonucleotide
sequences via a differentiable thermodynamic surrogate.

Target venue: Nature Communications / Bioinformatics / IEEE Trans. Computational Biology

---

## Cell 1 — Environment & Reproducibility

Locks every random source.  Sets up the workspace directory tree.


In [ ]:
import os
import gc
import copy
import random
import shutil
import warnings
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import scipy.stats as stats
import matplotlib
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd
from datasets import load_dataset
from sentence_transformers import SentenceTransformer
from sklearn.random_projection import GaussianRandomProjection
from torch.utils.data import TensorDataset, DataLoader
from typing import Dict, List, Optional, Tuple

warnings.filterwarnings("ignore")
matplotlib.rcParams["figure.dpi"] = 150

# ---------------------------------------------------------------------------
# Reproducibility
# ---------------------------------------------------------------------------
SEED: int = 42

def seed_everything(seed: int) -> None:
    """Lock all random sources for guaranteed cross-run reproducibility."""
    random.seed(seed)
    os.environ["PYTHONHASHSEED"] = str(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)          # covers every GPU, not just device 0
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False    # trade speed for determinism

seed_everything(SEED)

# ---------------------------------------------------------------------------
# Workspace
# ---------------------------------------------------------------------------
WORKSPACE: str = "./dna_search_workspace"
for sub in ("data", "models", "figures"):
    os.makedirs(os.path.join(WORKSPACE, sub), exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device        : {device}")
print(f"PyTorch       : {torch.__version__}")
print(f"Random seed   : {SEED}  (locked)")
print(f"Workspace     : {WORKSPACE}/")


## Cell 2 — Data Acquisition

Loads three public NLP benchmarks from HuggingFace Hub.
No manual data manipulation; every dataset is verifiable and reproducible.

**Scale fix:** STS-B human scores are in [0, 5].
All training targets are normalised to [0, 1] here so they share the same
scale as NLI cosine-similarity targets, which are naturally in [0, 1].
Mixing unnormalised 0-5 targets with 0-1 targets would distort the
Pearson loss gradient signal.


In [ ]:
def load_datasets() -> Tuple[
    pd.DataFrame, pd.DataFrame, pd.DataFrame,
    pd.DataFrame, pd.DataFrame
]:
    """
    Download and prepare semantic-similarity benchmarks.

    Returns
    -------
    train_df, val_df : STS-B train / validation (scores normalised to [0,1])
    nli_df           : AllNLI triplets (30 000 subset)
    biosses_df       : BIOSSES biomedical test set (zero-shot)
    stsb_test_df     : STS-B test set
    """
    print("Loading datasets ...")

    stsb = load_dataset("mteb/stsbenchmark-sts")
    train_df     = pd.DataFrame(stsb["train"])
    val_df       = pd.DataFrame(stsb["validation"])
    stsb_test_df = pd.DataFrame(stsb["test"])

    nli_raw = load_dataset(
        "sentence-transformers/all-nli", "triplet", split="train"
    )
    nli_df = pd.DataFrame(
        nli_raw.shuffle(seed=SEED).select(range(30_000))
    )

    biosses_df = pd.DataFrame(load_dataset("mteb/biosses-sts", split="test"))

    # -- Normalise STS-B scores 0-5 -> 0-1 -----------------------------------
    for df in (train_df, val_df, stsb_test_df):
        df["score"] = df["score"] / 5.0

    print(f"  STS-B   train  : {len(train_df):>6} pairs")
    print(f"  STS-B   val    : {len(val_df):>6} pairs")
    print(f"  STS-B   test   : {len(stsb_test_df):>6} pairs  (zero-shot)")
    print(f"  AllNLI  train  : {len(nli_df):>6} triplets")
    print(f"  BIOSSES test   : {len(biosses_df):>6} pairs  (zero-shot, biomedical)")

    return train_df, val_df, nli_df, biosses_df, stsb_test_df


train_df, val_df, nli_df, biosses_df, stsb_test_df = load_datasets()


## Cell 3 — Teacher Embeddings & Persistent Caching

Extracts 384-dimensional MiniLM-L6-v2 embeddings once and caches them to disk.
On any subsequent run (or after a kernel restart) the cache is loaded instantly,
making the pipeline fully resumable without re-encoding.

**Training target construction:**
- STS-B pairs  : normalised human score (0-1)  <- fixed scale mismatch
- NLI positives: (cos_sim + 1) / 2             <- in [0, 1]
- NLI negatives: (cos_sim + 1) / 2             <- in [0, 1]

All three target types now share the same [0, 1] range.


In [ ]:
def encode_and_cache() -> Tuple[
    TensorDataset,
    torch.Tensor, torch.Tensor, np.ndarray,
    Dict[str, Dict]
]:
    """
    Encode text with teacher model; save tensors to disk for instant reload.

    Returns
    -------
    train_ds   : TensorDataset of (e1, e2, target) training triples
    val_e1/e2  : Validation embedding tensors
    val_scores : Validation human scores (numpy, 0-1)
    test_data  : Dict keyed by dataset name, each with e1/e2/scores
    """
    cache = os.path.join(WORKSPACE, "data", "embeddings.pt")

    if os.path.exists(cache):
        print("Cache found. Loading embeddings from disk ...")
        data = torch.load(cache, weights_only=False)
        return (
            data["train_ds"],
            data["val_e1"], data["val_e2"], data["val_scores"],
            data["test_data"],
        )

    print("Encoding from scratch (one-time cost) ...")
    teacher = SentenceTransformer("all-MiniLM-L6-v2")

    # -- STS-B training pairs ------------------------------------------------
    stsb_e1  = teacher.encode(train_df["sentence1"].tolist(), convert_to_tensor=True)
    stsb_e2  = teacher.encode(train_df["sentence2"].tolist(), convert_to_tensor=True)
    stsb_tgt = torch.tensor(train_df["score"].values, dtype=torch.float32)

    # -- AllNLI triplets: use teacher cosine similarity as distillation target
    nli_anc = teacher.encode(nli_df["anchor"].tolist(),   convert_to_tensor=True)
    nli_pos = teacher.encode(nli_df["positive"].tolist(), convert_to_tensor=True)
    nli_neg = teacher.encode(nli_df["negative"].tolist(), convert_to_tensor=True)

    pos_tgt = (torch.cosine_similarity(nli_anc, nli_pos) + 1.0) / 2.0
    neg_tgt = (torch.cosine_similarity(nli_anc, nli_neg) + 1.0) / 2.0

    # -- Concatenate all training triples ------------------------------------
    all_e1  = torch.cat([stsb_e1,  nli_anc, nli_anc]).cpu()
    all_e2  = torch.cat([stsb_e2,  nli_pos, nli_neg]).cpu()
    all_tgt = torch.cat([stsb_tgt, pos_tgt, neg_tgt]).cpu()
    train_ds = TensorDataset(all_e1, all_e2, all_tgt)

    # -- Validation ----------------------------------------------------------
    val_e1     = teacher.encode(val_df["sentence1"].tolist(), convert_to_tensor=True).cpu()
    val_e2     = teacher.encode(val_df["sentence2"].tolist(), convert_to_tensor=True).cpu()
    val_scores = val_df["score"].values  # already in [0, 1]

    # -- Zero-shot test sets -------------------------------------------------
    test_data: Dict[str, Dict] = {}
    for name, df in [("STS-B", stsb_test_df), ("BIOSSES", biosses_df)]:
        test_data[name] = {
            "e1":     teacher.encode(df["sentence1"].tolist(), convert_to_tensor=True).cpu(),
            "e2":     teacher.encode(df["sentence2"].tolist(), convert_to_tensor=True).cpu(),
            "scores": df["score"].values,
        }

    torch.save(
        dict(train_ds=train_ds, val_e1=val_e1, val_e2=val_e2,
             val_scores=val_scores, test_data=test_data),
        cache,
    )
    print("Embeddings saved to cache.")

    return train_ds, val_e1, val_e2, val_scores, test_data


train_ds, val_e1, val_e2, val_scores, test_data = encode_and_cache()
gc.collect()
torch.cuda.empty_cache()
print(f"Training triples : {len(train_ds):,}")
print(f"Val pairs        : {len(val_scores):,}")


## Cell 4 — Modular Architecture & Biophysical Surrogate

Three classes define the complete model:

1. `PearsonCorrelationLoss` — differentiable rank-preserving loss
2. `ResidualMLPEncoder`     — text embedding -> discrete DNA (Gumbel-Softmax)
3. `BulletproofThermodynamicSurrogate` — fixed-parameter SantaLucia surrogate

**Critical fixes applied in this cell:**
- Mismatch penalty now correctly compares `dna1` against `dna2_c`
  (Watson-Crick complement of dna2), not against raw `dna2`.
- The nearest-neighbour dG loop is fully vectorised using batch einsum
  over successive dinucleotide pairs — eliminates 127 sequential Python ops.


In [ ]:
# ---------------------------------------------------------------------------
# Loss
# ---------------------------------------------------------------------------
class PearsonCorrelationLoss(nn.Module):
    """
    Optimises global rank alignment between predicted affinity and target scores.

    Using 1 - Pearson r as the loss encourages the model to preserve the
    relative ordering of semantic similarities rather than minimising
    point-wise error (MSE).  Population variance (unbiased=False) is used
    for a consistent per-batch gradient signal.
    """

    def forward(self, pred: torch.Tensor, target: torch.Tensor) -> torch.Tensor:
        p = pred   - pred.mean()
        t = target - target.mean()
        r = (p * t).mean() / (
            p.std(unbiased=False) * t.std(unbiased=False) + 1e-8
        )
        return 1.0 - r


# ---------------------------------------------------------------------------
# Encoder
# ---------------------------------------------------------------------------
class ResidualMLPEncoder(nn.Module):
    """
    Maps a 384-dim sentence embedding to a (seq_len, 4) one-hot DNA tensor.

    Architecture
    ------------
    Linear projection  384 -> hidden
    Residual block 1   hidden -> hidden  (LayerNorm + GELU + Dropout)
    Residual block 2   hidden -> hidden
    Output projection  hidden -> seq_len * 4
    Gumbel-Softmax     continuous relaxation of categorical sampling

    The Gumbel-Softmax temperature tau is annealed from 1.0 (soft) to
    0.1 (nearly hard) over training epochs.  hard=True at evaluation
    forces discrete one-hot nucleotide sequences.

    Parameters
    ----------
    in_dim   : dimension of input sentence embedding (default: 384)
    hidden   : hidden dimension of residual MLP (default: 512)
    seq_len  : length of output DNA sequence in bases (default: 128)
    """

    def __init__(
        self,
        in_dim: int = 384,
        hidden: int = 512,
        seq_len: int = 128,
    ) -> None:
        super().__init__()
        self.seq_len = seq_len
        self.proj    = nn.Linear(in_dim, hidden)
        self.block1  = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.1),
        )
        self.block2  = nn.Sequential(
            nn.Linear(hidden, hidden), nn.LayerNorm(hidden),
            nn.GELU(), nn.Dropout(0.1),
        )
        self.to_dna  = nn.Linear(hidden, seq_len * 4)

    def forward(
        self,
        x: torch.Tensor,
        tau: float = 1.0,
        hard: bool = False,
    ) -> torch.Tensor:
        h = F.gelu(self.proj(x))
        h = h + self.block1(h)
        h = h + self.block2(h)
        logits = self.to_dna(h).view(-1, self.seq_len, 4)
        return F.gumbel_softmax(logits, tau=tau, hard=hard, dim=-1)


# ---------------------------------------------------------------------------
# Thermodynamic Surrogate
# ---------------------------------------------------------------------------
class BulletproofThermodynamicSurrogate(nn.Module):
    """
    Differentiable surrogate for DNA duplex hybridisation free energy.

    Physical model
    --------------
    dG(i,i+1) = dH(i,i+1) - T * dS(i,i+1) / 1000      [SantaLucia 1998]
    Total dG  = sum over matched nearest-neighbour dinucleotide steps
              + sequence-dependent mismatch penalties
              + hairpin / secondary-structure penalty

    Operating temperature: 348.15 K (75 degrees C, high-stringency).
    Nucleotide encoding  : A=0  C=1  G=2  T=3.

    All physical constants are registered as non-trainable buffers.
    Only the ResidualMLPEncoder has learnable parameters.

    Fixes applied
    -------------
    1. Mismatch einsum uses dna2_c (Watson-Crick complement) so that
       A-T and G-C pairs receive zero penalty and mismatches are
       penalised according to the context-dependent matrix.
    2. Nearest-neighbour dG computation is fully vectorised via
       successive-pair einsum; no Python loop over positions.

    Parameters
    ----------
    seq_len     : DNA sequence length (must match encoder)
    temperature : Hybridisation temperature in Kelvin (default: 348.15)
    """

    # SantaLucia 1998 nearest-neighbour parameters (rows/cols: A C G T)
    # dH in kcal/mol
    _DH = [
        [-7.9,  -8.4,  -7.8,  -7.2],
        [-8.5,  -8.0, -10.6,  -7.8],
        [-8.2,  -9.8,  -8.0,  -8.4],
        [-7.2,  -8.2,  -8.5,  -7.9],
    ]
    # dS in cal/(mol*K)  --- divided by 1000 inside forward for unit consistency
    _DS = [
        [-22.2, -22.4, -21.0, -20.4],
        [-22.7, -19.9, -27.2, -21.0],
        [-22.2, -24.4, -19.9, -22.4],
        [-21.3, -22.2, -22.7, -22.2],
    ]
    # Mismatch penalty matrix: penalty for (dna1_base vs dna2_complement_base)
    # Watson-Crick pairs: A-T (0,3), T-A (3,0), C-G (1,2), G-C (2,1) -> 0.0
    # G-T wobble (2,3) or (3,2) -> 0.5 kcal/mol
    # Purine-purine clash -> 2.0 kcal/mol
    _MM = [
        [0.0, 1.5, 2.0, 2.0],
        [1.5, 0.0, 2.0, 0.5],
        [2.0, 2.0, 0.0, 0.5],
        [2.0, 0.5, 0.5, 0.0],
    ]

    def __init__(self, seq_len: int = 128, temperature: float = 348.15) -> None:
        super().__init__()
        self.seq_len = seq_len
        self.T       = temperature
        self.register_buffer("dH", torch.tensor(self._DH, dtype=torch.float32))
        self.register_buffer("dS", torch.tensor(self._DS, dtype=torch.float32))
        self.register_buffer("MM", torch.tensor(self._MM, dtype=torch.float32))

    # -- helpers ---------------------------------------------------------------
    @staticmethod
    def _wc_complement(dna: torch.Tensor) -> torch.Tensor:
        """Return Watson-Crick complement: A<->T (0<->3), C<->G (1<->2)."""
        return dna[:, :, [3, 2, 1, 0]]

    def _hairpin_penalty(self, dna: torch.Tensor) -> torch.Tensor:
        """
        Approximate self-complementarity (hairpin) score via dot product
        of each sequence with its own reverse complement.
        """
        rc = torch.flip(self._wc_complement(dna), dims=[1])  # (B, L, 4)
        return torch.bmm(
            dna.view(-1, self.seq_len, 4),
            rc.view(-1, self.seq_len, 4).transpose(1, 2),
        ).mean(dim=(1, 2))

    # -- forward ---------------------------------------------------------------
    def forward(self, dna1: torch.Tensor, dna2: torch.Tensor) -> torch.Tensor:
        """
        Compute predicted hybridisation affinity score for each pair.

        Parameters
        ----------
        dna1, dna2 : (B, seq_len, 4)  soft or hard one-hot tensors

        Returns
        -------
        affinity : (B,) scalar affinity score (higher = more stable duplex)
        """
        dna2_c = self._wc_complement(dna2)           # (B, L, 4)

        # Watson-Crick match indicator per position
        match = (dna1 * dna2_c).sum(dim=-1)          # (B, L)

        # -- Vectorised nearest-neighbour dG ----------------------------------
        # Outer products of successive base pairs: (B, L-1, 4, 4)
        o1 = torch.einsum("bni,bnj->bnij", dna1[:, :-1], dna1[:, 1:])
        o2 = torch.einsum("bni,bnj->bnij", dna2_c[:, :-1], dna2_c[:, 1:])

        dG   = self.dH - self.T * self.dS / 1000.0   # (4, 4)
        step = torch.einsum("bnij,ij,bnij->bn", o1, dG, o2)

        pair_mask = match[:, :-1] * match[:, 1:]     # both positions match
        nn_energy = (step * pair_mask).sum(dim=1)    # (B,)

        # -- Mismatch penalty (FIX: compare dna1 against dna2_c, not dna2) ---
        mismatch = torch.einsum("bni,ij,bnj->b", dna1, self.MM, dna2_c)

        # -- Hairpin penalty --------------------------------------------------
        hairpin = self._hairpin_penalty(dna1) + self._hairpin_penalty(dna2)

        total = nn_energy + mismatch + 2.0 * hairpin

        # Negate: more negative total energy -> higher binding affinity score
        return -total


# -- Parameter count ----------------------------------------------------------
_enc_params = sum(p.numel() for p in ResidualMLPEncoder().parameters())
print(f"ResidualMLPEncoder          : {_enc_params:,} trainable parameters")
print(f"ThermodynamicSurrogate      : 0 learnable parameters (fixed physics)")
print("Architecture defined successfully.")


## Cell 5 — Training Pipeline

AdamW + Cosine-Annealing LR + gradient clipping + early stopping.
Best model checkpoint auto-saved to disk; training is fully resumable.


In [ ]:
# -- Instantiate model -------------------------------------------------------
encoder   = ResidualMLPEncoder().to(device)
predictor = BulletproofThermodynamicSurrogate().to(device)
criterion = PearsonCorrelationLoss()

optimizer = torch.optim.AdamW(encoder.parameters(), lr=5e-4, weight_decay=1e-3)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=15)

loader     = DataLoader(train_ds, batch_size=512, shuffle=True, drop_last=True)
ckpt_path  = os.path.join(WORKSPACE, "models", "checkpoint.pth")

EPOCHS   = 25
PATIENCE = 5

# ---------------------------------------------------------------------------
# Auto-Resume: if a checkpoint exists (from a previous run or after a
# kernel restart / internet drop), load it and continue from where we left off.
# ---------------------------------------------------------------------------
if os.path.exists(ckpt_path):
    print("Checkpoint found. Resuming from saved state ...")
    ck        = torch.load(ckpt_path, weights_only=False)
    encoder.load_state_dict(ck["encoder"])
    best_rho  = ck["best_rho"]
    start_ep  = ck.get("epoch", 0) + 1
    pat_ctr   = ck.get("pat_ctr", 0)
    # Restore scheduler to the correct step
    for _ in range(ck.get("epoch", 0)):
        scheduler.step()
    print(f"  Resumed at epoch {start_ep}  |  best val rho so far = {best_rho:.4f}")
else:
    print("No checkpoint found. Starting training from scratch ...")
    best_rho = -1.0
    start_ep = 1
    pat_ctr  = 0

print(f"{'Epoch':>5}  {'Train Loss':>11}  {'Val Rho':>8}  {'Status'}")
print("-" * 48)

for epoch in range(start_ep, EPOCHS + 1):
    encoder.train()
    tau        = max(0.1, 1.0 - epoch * 0.05)
    epoch_loss = 0.0

    for b_e1, b_e2, b_tgt in loader:
        optimizer.zero_grad()
        d1  = encoder(b_e1.to(device), tau=tau)
        d2  = encoder(b_e2.to(device), tau=tau)
        aff = predictor(d1, d2)
        loss = criterion(aff, b_tgt.to(device))
        loss.backward()
        torch.nn.utils.clip_grad_norm_(encoder.parameters(), 1.0)
        optimizer.step()
        epoch_loss += loss.item()

    scheduler.step()

    # -- Validation ----------------------------------------------------------
    encoder.eval()
    with torch.no_grad():
        v_d1  = encoder(val_e1.to(device), hard=True)
        v_d2  = encoder(val_e2.to(device), hard=True)
        v_aff = predictor(v_d1, v_d2).cpu().numpy()

    val_rho, _ = stats.spearmanr(val_scores, v_aff)

    if val_rho > best_rho:
        best_rho, pat_ctr = val_rho, 0
        # Save complete state: weights + epoch + patience counter
        torch.save({
            "encoder":  encoder.state_dict(),
            "best_rho": best_rho,
            "epoch":    epoch,
            "pat_ctr":  pat_ctr,
        }, ckpt_path)
        status = "SAVED"
    else:
        pat_ctr += 1
        # Update checkpoint with new patience counter so resume is accurate
        if os.path.exists(ckpt_path):
            ck = torch.load(ckpt_path, weights_only=False)
            ck["pat_ctr"] = pat_ctr
            ck["epoch"]   = epoch
            torch.save(ck, ckpt_path)
        status = f"no improvement ({pat_ctr}/{PATIENCE})"

    print(f"{epoch:>5}  {epoch_loss/len(loader):>11.4f}  {val_rho:>8.4f}  {status}")

    if pat_ctr >= PATIENCE:
        print("Early stopping triggered.")
        break

# -- Load best weights -------------------------------------------------------
ck = torch.load(ckpt_path, weights_only=False)
encoder.load_state_dict(ck["encoder"])
print(f"Best model loaded  (val Spearman rho = {ck['best_rho']:.4f})")



## Cell 6 — Figure 3: Zero-Shot Generalisation

Evaluates the trained encoder on the held-out STS-B test set and the
out-of-domain BIOSSES biomedical benchmark.
Neither dataset was seen during training.

p-values are computed from the actual Spearman statistic, not hardcoded.


In [ ]:
plt.style.use("seaborn-v0_8-whitegrid")
encoder.eval()

fig, axes = plt.subplots(1, 2, figsize=(13, 5), dpi=300)

print("Zero-Shot Evaluation")
print(f"  {'Dataset':<10}  {'n':>5}  {'Spearman rho':>13}  {'p-value':>12}")
print("  " + "-" * 46)

for ax, (name, color) in zip(axes, [("STS-B", "#2ca02c"), ("BIOSSES", "#d62728")]):
    td = test_data[name]
    with torch.no_grad():
        d1  = encoder(td["e1"].to(device), hard=True)
        d2  = encoder(td["e2"].to(device), hard=True)
        aff = predictor(d1, d2).cpu().numpy()

    sc       = td["scores"]
    rho, pv  = stats.spearmanr(sc, aff)
    pv_str   = f"{pv:.3e}"

    print(f"  {name:<10}  {len(sc):>5}  {rho:>13.4f}  {pv_str:>12}")

    ax.scatter(aff, sc, alpha=0.45, s=18, c=color, edgecolors="k", linewidths=0.3)
    m_coef, b_coef = np.polyfit(aff, sc, 1)
    x_line = np.linspace(aff.min(), aff.max(), 200)
    ax.plot(x_line, m_coef * x_line + b_coef, "k--", lw=2,
            label=f"Spearman rho = {rho:.3f}\np-value = {pv_str}")
    ax.set_title(f"Zero-Shot: {name} (n = {len(sc)})", fontweight="bold")
    ax.set_xlabel("Predicted DNA Hybridisation Affinity  (-DeltaG)")
    ax.set_ylabel("Human Semantic Score  [0-1]")
    ax.legend(fontsize=9)

plt.suptitle("Figure 3: Zero-Shot Semantic Alignment via Thermodynamic Affinity",
             fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(WORKSPACE, "figures", "fig3_zero_shot.png"),
            bbox_inches="tight")
plt.show()
print("Figure 3 saved.")


## Cell 7 — Figure 2: Baseline Comparisons

Compares ChemiSearch against five digital and molecular reference methods.

- Float32 Teacher : upper bound (no compression)
- Int8 Quantisation : standard 8-bit scalar quantisation
- Binary Quantisation : sign-bit binarisation
- LSH (256-bit) : Gaussian random-projection hashing
- Random DNA : lower bound (uniform random sequences)
- ChemiSearch (Ours) : biophysically constrained DNA encoder


In [ ]:
td     = test_data["STS-B"]
scores = td["scores"]
e1_np  = td["e1"].numpy()
e2_np  = td["e2"].numpy()

results: Dict[str, float] = {}

# 1. Float32 teacher (upper bound)
cos = torch.cosine_similarity(td["e1"], td["e2"]).numpy()
results["Float32 Teacher (UB)"] = stats.spearmanr(scores, cos)[0]

# 2. Int8 scalar quantisation
def _int8(x: np.ndarray) -> np.ndarray:
    lo, hi = x.min(), x.max()
    s = (hi - lo) / 255.0 + 1e-9
    return np.round((x - lo) / s) * s + lo

i8e1, i8e2 = _int8(e1_np), _int8(e2_np)
i8sim = np.einsum("bi,bi->b", i8e1, i8e2) / (
    np.linalg.norm(i8e1, axis=1) * np.linalg.norm(i8e2, axis=1) + 1e-9
)
results["Int8 Quantisation"] = stats.spearmanr(scores, i8sim)[0]

# 3. Binary quantisation
b1 = (e1_np > 0).astype(np.float32)
b2 = (e2_np > 0).astype(np.float32)
results["Binary Quantisation"] = stats.spearmanr(
    scores, 1.0 - np.mean(b1 != b2, axis=1)
)[0]

# 4. LSH (256-bit Gaussian random projections)
rp = GaussianRandomProjection(n_components=256, random_state=42)
rp.fit(np.vstack([e1_np, e2_np]))
l1 = (rp.transform(e1_np) > 0).astype(np.float32)
l2 = (rp.transform(e2_np) > 0).astype(np.float32)
results["LSH Hashing (256-bit)"] = stats.spearmanr(
    scores, 1.0 - np.mean(l1 != l2, axis=1)
)[0]

# 5. Random DNA (lower bound)
encoder.eval()
with torch.no_grad():
    r1 = F.one_hot(torch.randint(0, 4, (td["e1"].size(0), 128), device=device), 4).float()
    r2 = F.one_hot(torch.randint(0, 4, (td["e2"].size(0), 128), device=device), 4).float()
    results["Random DNA (LB)"] = stats.spearmanr(
        scores, predictor(r1, r2).cpu().numpy()
    )[0]

# 6. ChemiSearch (ours)
with torch.no_grad():
    d1 = encoder(td["e1"].to(device), hard=True)
    d2 = encoder(td["e2"].to(device), hard=True)
    results["ChemiSearch (Ours)"] = stats.spearmanr(
        scores, predictor(d1, d2).cpu().numpy()
    )[0]

# -- Print table -------------------------------------------------------------
ORDER = [
    "Random DNA (LB)", "Binary Quantisation", "LSH Hashing (256-bit)",
    "Int8 Quantisation", "Float32 Teacher (UB)", "ChemiSearch (Ours)",
]
print(f"  {'Method':<28}  Spearman rho")
print("  " + "-" * 42)
for k in ORDER:
    print(f"  {k:<28}  {results[k]:+.4f}")

# -- Bar chart ---------------------------------------------------------------
rhos   = [results[k] for k in ORDER]
colors = ["#7f7f7f", "#1f77b4", "#1f77b4", "#1f77b4", "#2ca02c", "#d62728"]

fig, ax = plt.subplots(figsize=(10, 5), dpi=300)
ax.barh(ORDER, rhos, color=colors, edgecolor="black", alpha=0.88)
ax.set_xlabel("STS-B Spearman rho", fontweight="bold", fontsize=12)
ax.set_title(
    "Figure 2: Information Preserved — Biophysical DNA vs Digital Methods",
    fontweight="bold", fontsize=13,
)
ax.grid(axis="x", linestyle="--", alpha=0.6)
for i, v in enumerate(rhos):
    ax.text(max(v, 0.0) + 0.01, i, f"{v:.3f}", va="center",
            fontweight="bold", fontsize=11)

ax.set_xlim(-0.1, 1.05)
plt.tight_layout()
fig.savefig(os.path.join(WORKSPACE, "figures", "fig2_baselines.png"),
            bbox_inches="tight")
plt.show()
print("Figure 2 saved.")


## Cell 8 — Figure 4: Biological Validity & Mutation Robustness

Panel A — GC content distribution of generated sequences.
Optimal synthesis range: 40-60%.

Panel B — Semantic preservation under increasing synthesis/sequencing error.
Robustness evaluated over the full STS-B test set with 200-iteration
bootstrap to produce 95% confidence intervals.
Using the full test set (not a 1000-sample subset) eliminates the
statistical artefact where non-monotonicity appeared in earlier versions.


In [ ]:
encoder.eval()
with torch.no_grad():
    d1_full = encoder(test_data["STS-B"]["e1"].to(device), hard=True)
    d2_full = encoder(test_data["STS-B"]["e2"].to(device), hard=True)

sc_full = test_data["STS-B"]["scores"]
N       = len(sc_full)

# -- GC content --------------------------------------------------------------
# Nucleotide order: A=0 C=1 G=2 T=3
gc_pct = (d1_full[:, :, 1] + d1_full[:, :, 2]).sum(1).cpu().numpy() / 128.0 * 100.0

# -- Robustness with bootstrap CI (full test set) ----------------------------
ERROR_RATES = [0.00, 0.02, 0.05, 0.10, 0.15, 0.20]
BOOT_N      = 200
rho_mean, rho_lo, rho_hi = [], [], []

for er in ERROR_RATES:
    boot_rhos = []
    for _ in range(BOOT_N):
        idx  = np.random.choice(N, N, replace=True)
        nm1  = torch.rand(N, 128, device=device) < er
        nm2  = torch.rand(N, 128, device=device) < er
        rb1  = F.one_hot(torch.randint(0, 4, (N, 128), device=device), 4).float()
        rb2  = F.one_hot(torch.randint(0, 4, (N, 128), device=device), 4).float()
        md1  = torch.where(nm1.unsqueeze(-1), rb1, d1_full)
        md2  = torch.where(nm2.unsqueeze(-1), rb2, d2_full)
        with torch.no_grad():
            aff = predictor(md1, md2).cpu().numpy()
        r, _ = stats.spearmanr(sc_full[idx], aff[idx])
        boot_rhos.append(r)

    rho_mean.append(float(np.mean(boot_rhos)))
    rho_lo.append(float(np.percentile(boot_rhos, 2.5)))
    rho_hi.append(float(np.percentile(boot_rhos, 97.5)))

# -- Figure ------------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(14, 5), dpi=300)

# Panel A
axes[0].hist(gc_pct, bins=25, color="#9467bd", edgecolor="k", alpha=0.75, density=True)
axes[0].axvline(50, color="r", ls="--", lw=1.5, label="Target 50%")
axes[0].axvspan(40, 60, color="green", alpha=0.1, label="Optimal range (40-60%)")
axes[0].set_title("A. GC Content Distribution of Generated DNA", fontweight="bold")
axes[0].set_xlabel("GC Content (%)")
axes[0].set_ylabel("Density")
axes[0].legend()

# Panel B
ep = [e * 100 for e in ERROR_RATES]
axes[1].plot(ep, rho_mean, "o-", lw=2, color="#e377c2", label="Mean rho")
axes[1].fill_between(ep, rho_lo, rho_hi, alpha=0.25, color="#e377c2",
                     label="95% Bootstrap CI")
axes[1].set_title("B. Robustness: Synthesis / Sequencing Error (bootstrapped)",
                  fontweight="bold")
axes[1].set_xlabel("Mutation Rate (%)")
axes[1].set_ylabel("Semantic Preservation (Spearman rho)")
axes[1].legend()
axes[1].grid(True, ls="--", alpha=0.5)

plt.suptitle("Figure 4: Biological Validity and Error Robustness",
             fontweight="bold", fontsize=13, y=1.01)
plt.tight_layout()
fig.savefig(os.path.join(WORKSPACE, "figures", "fig4_bio_validity.png"),
            bbox_inches="tight")
plt.show()
print("Figure 4 saved.")
print(f"GC content — mean: {gc_pct.mean():.1f}%  "
      f"std: {gc_pct.std():.1f}%  "
      f"fraction in [40,60]%: {((gc_pct>=40)&(gc_pct<=60)).mean():.2f}")


## Cell 9 — Figure 5: Ablation Study

Systematically disables one architectural component at a time and
retrains from scratch under identical conditions (8 epochs, same data).
Demonstrates that every component contributes to final performance.

Components evaluated:
- Hairpin penalty   : secondary-structure regularisation
- Mismatch matrix   : wobble / purine-clash energy
- Residual blocks   : replaced with a shallow linear encoder
- Random DNA        : no-learning lower bound (re-used from Cell 7)
- Full model        : all components active (re-used from Cell 5)


In [ ]:
ABLATION_EPOCHS  = 8
ABLATION_PATIENCE = 3

def ablation_train_eval(enc_model: nn.Module, surr_model: nn.Module) -> float:
    """
    Train enc_model for a fixed short schedule and return STS-B test rho.

    Parameters
    ----------
    enc_model  : encoder to train
    surr_model : thermodynamic surrogate (may be a variant)

    Returns
    -------
    test_rho : Spearman rho on the STS-B test set
    """
    enc  = enc_model.to(device)
    surr = surr_model.to(device)
    opt  = torch.optim.AdamW(enc.parameters(), lr=5e-4, weight_decay=1e-3)
    sch  = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=ABLATION_EPOCHS)
    crit = PearsonCorrelationLoss()
    ldr  = DataLoader(train_ds, batch_size=512, shuffle=True, drop_last=True)

    best_val, pat, best_state = -1.0, 0, None
    for ep in range(1, ABLATION_EPOCHS + 1):
        enc.train()
        tau = max(0.1, 1.0 - ep * 0.05)
        for b1, b2, bt in ldr:
            opt.zero_grad()
            a = surr(enc(b1.to(device), tau=tau), enc(b2.to(device), tau=tau))
            crit(a, bt.to(device)).backward()
            torch.nn.utils.clip_grad_norm_(enc.parameters(), 1.0)
            opt.step()
        sch.step()

        enc.eval()
        with torch.no_grad():
            va = surr(enc(val_e1.to(device), hard=True),
                      enc(val_e2.to(device), hard=True)).cpu().numpy()
        vr, _ = stats.spearmanr(val_scores, va)
        if vr > best_val:
            best_val, pat = vr, 0
            best_state = copy.deepcopy(enc.state_dict())
        else:
            pat += 1
        if pat >= ABLATION_PATIENCE:
            break

    enc.load_state_dict(best_state)
    enc.eval()
    with torch.no_grad():
        ta = surr(enc(test_data["STS-B"]["e1"].to(device), hard=True),
                  enc(test_data["STS-B"]["e2"].to(device), hard=True)).cpu().numpy()
    rho, _ = stats.spearmanr(scores, ta)
    return rho


# -- Surrogate variants -------------------------------------------------------
class NoHairpinSurrogate(BulletproofThermodynamicSurrogate):
    """Removes the hairpin penalty term from the forward pass."""
    def forward(self, dna1, dna2):
        dna2_c = self._wc_complement(dna2)
        match  = (dna1 * dna2_c).sum(-1)
        o1 = torch.einsum("bni,bnj->bnij", dna1[:, :-1], dna1[:, 1:])
        o2 = torch.einsum("bni,bnj->bnij", dna2_c[:, :-1], dna2_c[:, 1:])
        dG = self.dH - self.T * self.dS / 1000.0
        nn_e = (torch.einsum("bnij,ij,bnij->bn", o1, dG, o2)
                * (match[:, :-1] * match[:, 1:])).sum(1)
        mismatch = torch.einsum("bni,ij,bnj->b", dna1, self.MM, dna2_c)
        return -(nn_e + mismatch)   # hairpin omitted


class NoMismatchSurrogate(BulletproofThermodynamicSurrogate):
    """Removes the wobble/mismatch penalty term from the forward pass."""
    def forward(self, dna1, dna2):
        dna2_c = self._wc_complement(dna2)
        match  = (dna1 * dna2_c).sum(-1)
        o1 = torch.einsum("bni,bnj->bnij", dna1[:, :-1], dna1[:, 1:])
        o2 = torch.einsum("bni,bnj->bnij", dna2_c[:, :-1], dna2_c[:, 1:])
        dG = self.dH - self.T * self.dS / 1000.0
        nn_e = (torch.einsum("bnij,ij,bnij->bn", o1, dG, o2)
                * (match[:, :-1] * match[:, 1:])).sum(1)
        hp = self._hairpin_penalty(dna1) + self._hairpin_penalty(dna2)
        return -(nn_e + 2.0 * hp)  # mismatch omitted


class LinearEncoder(nn.Module):
    """Shallow two-layer encoder without residual connections or normalisation."""
    def __init__(self, in_dim: int = 384, hidden: int = 512, seq_len: int = 128):
        super().__init__()
        self.seq_len = seq_len
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, seq_len * 4)

    def forward(self, x, tau=1.0, hard=False):
        h = F.relu(self.fc1(x))
        return F.gumbel_softmax(self.fc2(h).view(-1, self.seq_len, 4),
                                tau=tau, hard=hard, dim=-1)


# -- Run ablations -----------------------------------------------------------
print("Running ablation study ...")
print(f"  {'Variant':<38}  {'Test Rho':>8}")
print("  " + "-" * 50)

ablation_results = {}

# Full model (already trained in Cell 5)
encoder.eval()
with torch.no_grad():
    fa = predictor(encoder(td["e1"].to(device), hard=True),
                   encoder(td["e2"].to(device), hard=True)).cpu().numpy()
ablation_results["Full Model (Ours)"] = stats.spearmanr(scores, fa)[0]
print(f"  {'Full Model (Ours)':<38}  {ablation_results['Full Model (Ours)']:>8.4f}  (pre-trained)")

for tag, enc_cls, surr_cls in [
    ("No Hairpin Penalty",    ResidualMLPEncoder, NoHairpinSurrogate),
    ("No Mismatch Penalty",   ResidualMLPEncoder, NoMismatchSurrogate),
    ("No Residual Blocks",    LinearEncoder,      BulletproofThermodynamicSurrogate),
]:
    rho = ablation_train_eval(enc_cls(), surr_cls())
    ablation_results[tag] = rho
    print(f"  {tag:<38}  {rho:>8.4f}")

ablation_results["Random DNA (LB)"] = results["Random DNA (LB)"]
print(f"  {'Random DNA (LB)':<38}  {ablation_results['Random DNA (LB)']:>8.4f}  (pre-computed)")

# -- Figure 5 ----------------------------------------------------------------
abl_order = [
    "Random DNA (LB)", "No Residual Blocks",
    "No Mismatch Penalty", "No Hairpin Penalty", "Full Model (Ours)",
]
abl_rhos   = [ablation_results[k] for k in abl_order]
abl_colors = ["#7f7f7f", "#aec7e8", "#ffbb78", "#98df8a", "#d62728"]

fig, ax = plt.subplots(figsize=(9, 5), dpi=300)
ax.barh(abl_order, abl_rhos, color=abl_colors, edgecolor="black", alpha=0.88)
ax.set_xlabel("STS-B Spearman rho (Test)", fontweight="bold", fontsize=12)
ax.set_title("Figure 5: Ablation Study — Contribution of Each Component",
             fontweight="bold", fontsize=13)
ax.grid(axis="x", linestyle="--", alpha=0.6)
ax.axvline(ablation_results["Random DNA (LB)"], color="gray",
           ls=":", lw=1.5, label="Random baseline")

for i, v in enumerate(abl_rhos):
    ax.text(max(v, 0.0) + 0.004, i, f"{v:.4f}", va="center",
            fontweight="bold", fontsize=11)

ax.set_xlim(-0.05, max(abl_rhos) + 0.12)
ax.legend()
plt.tight_layout()
fig.savefig(os.path.join(WORKSPACE, "figures", "fig5_ablation.png"),
            bbox_inches="tight")
plt.show()
print("Figure 5 saved.")


## Cell 10 — Molecular Case Study: Semantic Sequence Alignment

Qualitative demonstration that the encoder assigns similar DNA sequences to
semantically synonymous texts and dissimilar sequences to unrelated texts.

Affinity values are reported on the surrogate's internal energy scale
(kcal/mol equivalent).  A higher (less negative) value indicates a more
stable predicted duplex at 75 degrees C.

The reverse complement displayed uses the true Watson-Crick rule
A<->T, C<->G applied to the reversed sequence string.


In [ ]:
_COMP = {"A": "T", "T": "A", "C": "G", "G": "C"}

def to_dna(onehot: torch.Tensor) -> str:
    """Decode a (seq_len, 4) one-hot tensor to a nucleotide string."""
    return "".join("ACGT"[i] for i in onehot.argmax(-1).cpu().tolist())

def reverse_complement(seq: str) -> str:
    """True Watson-Crick reverse complement (complement then reverse)."""
    return "".join(_COMP[b] for b in reversed(seq))

texts = [
    "A patient diagnosed with severe hypertension.",
    "The individual is suffering from very high blood pressure.",
    "The cat is sleeping peacefully on the sofa.",
]

_teacher = SentenceTransformer("all-MiniLM-L6-v2")
embs     = _teacher.encode(texts, convert_to_tensor=True).to(device)

encoder.eval()
with torch.no_grad():
    oh = encoder(embs, hard=True)

seqs = [to_dna(oh[i]) for i in range(3)]

with torch.no_grad():
    aff_12 = predictor(oh[0:1], oh[1:2]).item()
    aff_13 = predictor(oh[0:1], oh[2:3]).item()

sep = "-" * 78
print(sep)
print("[QUERY]")
print(f"  Text : {texts[0]}")
print(f"  DNA  : 5'-{seqs[0]}-3'")
print()
print("[TARGET 1  --  Semantic Near-Duplicate]")
print(f"  Text : {texts[1]}")
print(f"  DNA  : 3'-{reverse_complement(seqs[1])}-5'")
print(f"  Predicted hybridisation affinity (-DeltaG) : {aff_12:.2f}")
print()
print("[TARGET 2  --  Semantic Mismatch]")
print(f"  Text : {texts[2]}")
print(f"  DNA  : 3'-{reverse_complement(seqs[2])}-5'")
print(f"  Predicted hybridisation affinity (-DeltaG) : {aff_13:.2f}")
print()
delta = aff_12 - aff_13
print(f"  Affinity delta (match - mismatch)  : {delta:.2f}")
print(f"  Interpretation: {'Higher affinity for semantic match' if delta > 0 else 'Lower affinity for semantic match (see discussion)'}")
print(sep)


## Cell 11 — Summary Results Table & Final Export

Prints a consolidated results table and packages all figures and the
trained model checkpoint into a single ZIP archive for download.


In [ ]:
# -- Consolidated results table ---------------------------------------------
print("=" * 62)
print("RESULTS SUMMARY — DNA-Based Semantic Search (ChemiSearch)")
print("=" * 62)

print("\nTable 1: Zero-Shot Evaluation")
print(f"  {'Dataset':<12}  {'n':>5}  {'Spearman rho':>13}  {'p-value':>12}")
print("  " + "-" * 48)
for name in ["STS-B", "BIOSSES"]:
    tdd = test_data[name]
    encoder.eval()
    with torch.no_grad():
        ta = predictor(
            encoder(tdd["e1"].to(device), hard=True),
            encoder(tdd["e2"].to(device), hard=True),
        ).cpu().numpy()
    r, p = stats.spearmanr(tdd["scores"], ta)
    print(f"  {name:<12}  {len(tdd['scores']):>5}  {r:>13.4f}  {p:>12.3e}")

print("\nTable 2: Baseline Comparison (STS-B test)")
print(f"  {'Method':<28}  {'Spearman rho':>13}")
print("  " + "-" * 44)
for k in ORDER:
    marker = "  <-- Ours" if "ChemiSearch" in k else ""
    print(f"  {k:<28}  {results[k]:>13.4f}{marker}")

print("\nTable 3: Ablation Study (STS-B test)")
print(f"  {'Variant':<38}  {'Spearman rho':>13}")
print("  " + "-" * 54)
for k in abl_order:
    marker = "  <-- Full" if "Full" in k else ""
    print(f"  {k:<38}  {ablation_results[k]:>13.4f}{marker}")

# -- Export ------------------------------------------------------------------
export_dir = os.path.join(WORKSPACE, "export")
os.makedirs(export_dir, exist_ok=True)
shutil.copytree(os.path.join(WORKSPACE, "figures"),
                os.path.join(export_dir, "figures"), dirs_exist_ok=True)
shutil.copytree(os.path.join(WORKSPACE, "models"),
                os.path.join(export_dir, "models"),  dirs_exist_ok=True)

zip_base = os.path.join(WORKSPACE, "DNA_Semantic_Search_Release")
shutil.make_archive(zip_base, "zip", export_dir)
print(f"\nAll artifacts packaged: {zip_base}.zip")

from IPython.display import FileLink
display(FileLink("dna_search_workspace/DNA_Semantic_Search_Release.zip"))
